In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


In [ ]:
df_train = pd.read_csv('/kaggle/input/amazon-ml-dataset-2025/train.csv')

In [ ]:
df_train

In [ ]:
import pandas as pd
import os
from pathlib import Path

# Extract filenames from the image_link column
df_train["filename"] = df_train["image_link"].apply(lambda x: Path(x).name)

folder = "/kaggle/input/first-1000-images"
local_files = set(os.listdir(folder))

# Count how many match
matches = df_train["filename"].isin(local_files).sum()
print(f"{matches} / {len(df_train)} files found locally")


In [ ]:
df_train

In [ ]:
image_folder = '/kaggle/input/first-1000-images'
available_files = {f: os.path.join(image_folder, f) for f in os.listdir(image_folder)}
print(f"{len(available_files)} images available")


In [ ]:
import pandas as pd
from pathlib import Path
from PIL import Image
import os

def load_image_safe(filename):
    path = available_files.get(filename)
    if path and os.path.exists(path):
        try:
            return Image.open(path).convert("RGB")
        except Exception as e:
            print(f"⚠️ Error loading {filename}: {e}")
            return None
    else:
        return None

df_train["image_obj"] = df_train["filename"].apply(load_image_safe)


In [ ]:
df_train

In [ ]:
df_with_images = df_train[df_train["image_obj"].notna()].copy()
print(f"Kept {len(df_with_images)} rows with valid images")


In [ ]:
df_with_images



In [ ]:
df_with_images.columns

In [ ]:
cols_to_keep = ['catalog_content', 'price', 'image_obj']

In [ ]:
df_cols = df_with_images[cols_to_keep]

In [ ]:
df_cols

In [ ]:
df_cols['catalog_content']

In [ ]:
df_cols['catalog_content'][73897]

In [ ]:
import pandas as pd
import re

# --- Step 1: Define the parsing function ---
# This function takes the raw text string and returns a dictionary.
def parse_product_info(text_block):
    """
    Parses a product information string into a structured dictionary.
    """
    data = {
        'Item Name': None,
        'Bullet Points': None,
        'Value': None,
        'Unit': None
    }

    # Use regex to find and extract the item name
    item_name_match = re.search(r"Item Name: (.*)", text_block)
    if item_name_match:
        data['Item Name'] = item_name_match.group(1).strip()

    # Use regex to find all bullet points and join them into a single string
    bullet_points = re.findall(r"Bullet Point \d+: (.*)", text_block)
    if bullet_points:
        # Format the bullets for readability
        data['Bullet Points'] = "\n• ".join([bp.strip() for bp in bullet_points])
        data['Bullet Points'] = "• " + data['Bullet Points']


    # Use regex to find the value
    value_match = re.search(r"Value: (.*)", text_block)
    if value_match:
        # Convert value to a float for numerical operations
        data['Value'] = float(value_match.group(1).strip())

    # Use regex to find the unit
    unit_match = re.search(r"Unit: (.*)", text_block)
    if unit_match:
        data['Unit'] = unit_match.group(1).strip()

    return data


# --- Step 3: Apply the function and create new columns ---
# We apply the function to the 'raw_description' column.
# The result of .apply() is a Series of dictionaries.
# We convert this Series of dictionaries into a new DataFrame.
parsed_df = df_cols['catalog_content'].apply(parse_product_info).apply(pd.Series)

# You can also join this back to your original DataFrame if needed
# final_df = df.join(parsed_df)

# --- Step 4: Display the result ---
print(parsed_df)

In [ ]:
parsed_df.head(1)

In [ ]:
df_cols.head(1)

In [ ]:
merged_df = df_cols.join(parsed_df)


In [ ]:
merged_df.columns

In [ ]:
cols_to_use = ['price', 'image_obj', 'Item Name', 'Bullet Points', 'Value', 'Unit']

In [ ]:
df1 = merged_df[cols_to_use]

In [ ]:
df1.head(1)

In [ ]:
target_size = (224, 224)

df1['image_obj_resized'] = df1['image_obj'].apply(lambda img: img.resize(target_size))

print("\nDataFrame with Resized Image:")
print(df1)
print(f"\nResized Image Size: {df1.loc[73897, 'image_obj_resized'].size}")

In [ ]:
df2 = df1.drop(['image_obj'], axis=1)


In [ ]:
df2.info()

In [ ]:
df2.head(2)

In [ ]:
df3 = pd.read_csv('/kaggle/input/amazon-ml-dataset-2025/train.csv')

In [ ]:
df3.head()

In [ ]:
import re
import numpy as np # Import numpy for np.nan, the standard for missing numbers

def parse_product_info(text_block):
    """
    Parses a product information string into a structured dictionary.
    This version is more robust against formatting errors.
    """
    data = {
        'Item Name': None,
        'Bullet Points': None,
        'Value': np.nan, # Use np.nan as a default for missing numerical data
        'Unit': None
    }
    
    # Ensure the input is a string to avoid errors on non-string data
    if not isinstance(text_block, str):
        return data

    # Use regex to find and extract the item name
    item_name_match = re.search(r"Item Name: (.*)", text_block)
    if item_name_match:
        data['Item Name'] = item_name_match.group(1).strip()

    # Use regex to find all bullet points and join them into a single string
    bullet_points = re.findall(r"Bullet Point \d+: (.*)", text_block)
    if bullet_points:
        data['Bullet Points'] = "\n• ".join([bp.strip() for bp in bullet_points])
        data['Bullet Points'] = "• " + data['Bullet Points']

    # --- START OF CORRECTED SECTION ---

    # 1. Use a more specific regex to find a numerical value (digits and decimals)
    value_match = re.search(r"Value: ([\d.]+)", text_block)
    if value_match:
        try:
            # 2. Try to convert the captured group to a float
            data['Value'] = float(value_match.group(1).strip())
        except ValueError:
            # If conversion fails (e.g., malformed number like "1.2.3"),
            # it will remain np.nan.
            pass 
            
    # --- END OF CORRECTED SECTION ---

    # Use regex to find the unit
    unit_match = re.search(r"Unit: (.*)", text_block)
    if unit_match:
        data['Unit'] = unit_match.group(1).strip()

    return data

In [ ]:
parsed_df = df3['catalog_content'].apply(parse_product_info).apply(pd.Series)


In [ ]:
parsed_df.head()

In [ ]:
df4 = df3.join(parsed_df)

In [ ]:
df4.head(1)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# --- Plotting a Histogram with Seaborn ---
print("\nDisplaying Seaborn Histogram...")
plt.figure(figsize=(8, 5)) # Control the figure size
sns.histplot(data=df4, 
             x='price', 
             bins=20, 
             kde=True) # Adds a Kernel Density Estimate line
plt.title('Distribution of Product Prices (Seaborn)')
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
df3['log_price'] = np.log1p(df3['price'])


In [ ]:
df3

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
# --- Plotting a Histogram with Seaborn ---
print("\nDisplaying Seaborn Histogram...")
plt.figure(figsize=(8, 5)) # Control the figure size
sns.histplot(data=df3, 
             x='log_price', 
             bins=10, 
             kde=True) # Adds a Kernel Density Estimate line
plt.title('Distribution of Product Prices (Seaborn)')
plt.xlabel("Price")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()

In [ ]:
df3

In [ ]:
df1

In [ ]:
df1['log_price'] = np.log1p(df1['price'])


In [ ]:
df1.head()

In [ ]:
df1.info()

In [ ]:
df4.info()

In [ ]:
import pandas as pd
import numpy as np

df1['Bullet Points'].fillna('Unknown', inplace=True)


df_f = df1.dropna(subset=['Value'], inplace=True)



In [ ]:
df1.info()

In [ ]:
unique_units = df1['Unit'].unique()


In [ ]:
df4['log_price'] = np.log1p(df4['price'])


In [ ]:
unique_units3 = df4['Unit'].unique()


In [ ]:
unique_units3

In [ ]:
import pandas as pd
import numpy as np

def standardize_unit(unit):
    """
    Normalizes a unit string to a consistent, standardized format.
    """
    # Handle null values or non-string types first
    if pd.isna(unit) or not isinstance(unit, str):
        return 'Unknown'

    # Clean the input by making it lowercase and removing whitespace
    clean_unit = unit.lower().strip()

    # Handle empty or placeholder strings
    if not clean_unit or clean_unit in ['-', '---', 'none', 'product_weight']:
        return 'Unknown'

    # --- Check for units in a specific order to avoid conflicts ---

    # 1. Volume Units (check for "fl" before "oz")
    if 'fl' in clean_unit or 'fluid' in clean_unit:
        return 'Fluid Ounce'
    if 'liter' in clean_unit or 'ltr' in clean_unit:
        return 'Liter'
    if 'gal' in clean_unit:
        return 'Gallon'
    if 'ml' in clean_unit or 'millilitre' in clean_unit or 'mililitro' in clean_unit:
        return 'Milliliter'

    # 2. Weight Units
    if 'ounce' in clean_unit or 'oz' in clean_unit:
        return 'Ounce'
    if 'pound' in clean_unit or 'lb' in clean_unit:
        return 'Pound'
    if 'gram' in clean_unit or 'gr' in clean_unit:
        return 'Gram'
    if 'kg' in clean_unit:
        return 'Kilogram'

    # 3. Count-based Units
    if any(keyword in clean_unit for keyword in ['count', 'ct', 'each', 'piece', 'unit']):
        return 'Count'
    if 'pack' in clean_unit:
        return 'Pack'
    if 'bag' in clean_unit:
        return 'Bag'
    if 'bottle' in clean_unit:
        return 'Bottle'
    if 'can' in clean_unit:
        return 'Can'
    if 'box' in clean_unit:
        return 'Box'
    if any(keyword in clean_unit for keyword in ['jar', 'pouch', 'k-cup', 'capsule']):
        return 'Count'

    # Check if the string is just a number (which implies a count)
    if clean_unit.isdigit():
        return 'Count'

    # If no match is found after all checks, classify as Unknown
    return 'Unknown'

In [ ]:
df4['Unit_Standardized'] = df4['Unit'].apply(standardize_unit)

In [ ]:
df4['Unit_Standardized'].unique()

In [ ]:
unique_units

In [ ]:
df1['Unit_Standardized'] = df1['Unit'].apply(standardize_unit)

In [ ]:
df1['Unit_Standardized'].unique()

In [ ]:
df1['Unit_Standardized'].value_counts()

In [ ]:
df4['Unit_Standardized'].value_counts()

In [ ]:
df1.head(1)

In [ ]:
# ensure it's string/categorical
df1["Unit_Standardized"] = df1["Unit_Standardized"].astype("category")

# optional: treat NaN as its own bucket
df1["Unit_Standardized"] = df1["Unit_Standardized"].cat.add_categories(["<NA>"]).fillna("<NA>")

# make dummies (drop_first=False keeps all columns; set True to avoid dummy trap)
unit_ohe = pd.get_dummies(df1["Unit_Standardized"],
                          prefix="unit",
                          drop_first=False)

# join back
df_ohe = pd.concat([df1.drop(columns=["Unit_Standardized"]), unit_ohe], axis=1)


In [ ]:
df_ohe.head(1)

In [ ]:
df_ohe.columns

In [ ]:
col = ['Item Name', 'Bullet Points', 'Value', 'Unit',
       'image_obj_resized', 'log_price', 'unit_Count', 'unit_Fluid Ounce',
       'unit_Ounce', 'unit_Pound', 'unit_<NA>']

In [ ]:
df_ohe1 = df_ohe[col]

In [ ]:
df_ohe1.head(1)

In [ ]:
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm

# Import TensorFlow and the ResNet50 model
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import Model

# --- Step 1: Load the Pre-trained ResNet50 Model ---

# Load ResNet50 with pre-trained ImageNet weights
# include_top=False: This removes the final fully-connected classification layer.
# pooling='avg': This adds a Global Average Pooling layer to get a 1D vector.
base_model = ResNet50(weights='imagenet', include_top=False, pooling='avg')

# The output of this model is the 2048-dimensional feature vector
model = Model(inputs=base_model.input, outputs=base_model.output)


# --- Step 2: Create a Function to Process and Encode an Image ---

def encode_image(image_obj):
    """
    Takes a PIL Image, preprocesses it, and returns a 2048-dim vector.
    """
    try:
        # 1. Resize the image to the standard ResNet50 input size (224x224)
        image_resized = image_obj.resize((224, 224))
        
        # 2. Convert the image to a NumPy array
        image_array = np.array(image_resized)
        
        # 3. If the image is grayscale, convert it to RGB
        if image_array.ndim == 2:
            image_array = np.stack((image_array,) * 3, axis=-1)
            
        # 4. Add a batch dimension (model expects a batch of images)
        image_batch = np.expand_dims(image_array, axis=0)
        
        # 5. Preprocess the image for the ResNet50 model (scales pixels)
        preprocessed_image = preprocess_input(image_batch)
        
        # 6. Get the feature vector from the model
        feature_vector = model.predict(preprocessed_image, verbose=0)
        
        # 7. Flatten the vector to a 1D array
        return feature_vector.flatten()
        
    except Exception as e:
        print(f"Error processing image: {e}")
        # Return a vector of NaNs if an error occurs
        return np.full(2048, np.nan)



In [ ]:

print("Encoding images... This may take a while.")

tqdm.pandas()
df_ohe1['image_vector'] = df_ohe1['image_obj_resized'].progress_apply(encode_image)

print("\n--- DataFrame after Encoding ---")
#print(df)

# --- Verify the output shape ---
print(f"\nShape of the first vector: {df_ohe1['image_vector'].iloc[0].shape}")

In [ ]:
df_ohe1.head(1)

In [ ]:
df_ohe1.columns

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

df_ohe1['combined_text'] = df_ohe1['Item Name'].fillna('') + ' ' + df_ohe1['Bullet Points'].fillna('')

print(df_ohe1[['Item Name', 'combined_text']].head())

tfidf_vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')

tfidf_matrix = tfidf_vectorizer.fit_transform(df_ohe1['combined_text'])


# --- Step 3: View the Results ---

print("\n--- TF-IDF Matrix Shape ---")
# The shape will be (number of items, number of features/words)
print(tfidf_matrix.shape)

# You can get the list of words (features) the vectorizer learned
feature_names = tfidf_vectorizer.get_feature_names_out()
print("\n--- Sample of Learned Vocabulary (Features) ---")
print(feature_names[:20])

# The output is a sparse matrix. To see it as a regular array:
print("\n--- TF-IDF Matrix (Dense Array Representation) ---")
print(tfidf_matrix.toarray())

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names, index=df_ohe1.index)


# --- Step 2: Concatenate the original DataFrame with the new TF-IDF DataFrame ---

# Use pd.concat with axis=1 to join the columns side-by-side.
df_with_tfidf = pd.concat([df_ohe1, tfidf_df], axis=1)


In [ ]:
df_with_tfidf.head(1)

In [ ]:
import pandas as pd
import numpy as np


image_features_df = df_with_tfidf['image_vector'].apply(pd.Series)


# --- Step 2: Rename the new columns for clarity ---

# Create new column names like 'img_vec_0', 'img_vec_1', ...
image_features_df.columns = [f'img_vec_{i}' for i in range(image_features_df.shape[1])]


# --- Step 3: Concatenate the new features with the original DataFrame ---

# Use pd.concat with axis=1 to join them side-by-side.
df_expanded = pd.concat([df_with_tfidf, image_features_df], axis=1)


# --- Step 4: Drop the original 'image_vector' column ---

# This removes the now-redundant original column.
df_expanded.drop('image_vector', axis=1, inplace=True)


# --- Display the final result ---
print("\n--- Expanded DataFrame ---")
# .info() is a great way to see the new structure and column count
print(df_expanded.info())

print("\n--- First 5 Columns of Expanded DataFrame ---")
print(df_expanded.iloc[:, :5].head())

In [ ]:
df_expanded.info()

In [ ]:
y_mock = df_expanded['log_price']

In [ ]:
y_mock = y_mock.to_numpy()

In [ ]:
X_mock = df_expanded.drop('log_price', axis=1)

In [ ]:
type(X_mock)

In [ ]:
df_expanded1 = df_expanded.drop(['Item Name', 'Bullet Points', 'Unit', 'image_obj_resized', 'combined_text'], axis = 1)

In [ ]:
df_expanded1.info()

In [ ]:
df_expanded1.columns

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


n_samples = 899
n_features = 3055


# --- Step 1: Separate Features (X) and Target (y) ---
X = df_expanded1.drop('log_price', axis=1)
y = df_expanded1['log_price']


# --- Step 2: Split Data into Training and Testing Sets ---
# 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- Step 3: Scale the Feature Data ---
# Neural networks perform best when input features are scaled.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# --- Step 4: Build the Neural Network Model ---
model = Sequential([
    # Input layer: Must match the number of features
    Input(shape=(X_train_scaled.shape[1],)),
    
    # First hidden layer with 128 neurons and ReLU activation
    Dense(128, activation='relu'),
    
    # Second hidden layer with 64 neurons
    Dense(64, activation='relu'),
    
    # Dropout layer to prevent overfitting
    Dropout(0.2),
    
    # Third hidden layer with 32 neurons
    Dense(32, activation='relu'),
    
    # Output layer: A single neuron for a regression task (predicting one value)
    Dense(1)
])

# --- Step 5: Compile the Model ---
# For regression, 'mean_squared_error' is a common loss function.
# 'adam' is a robust optimizer.
# We also monitor 'mean_absolute_error' for an interpretable metric.
model.compile(optimizer='adam', 
              loss='mean_squared_error', 
              metrics=['mean_absolute_error'])

# Print a summary of the model's architecture
model.summary()



# --- Step 6: Train the Model ---
print("\nTraining the model...")
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=50,          # Number of times to iterate over the entire dataset
    batch_size=32,      # Number of samples per gradient update
    validation_data=(X_test_scaled, y_test), # Data to evaluate at the end of each epoch
    verbose=1           # Show a progress bar
)



# --- Step 7: Evaluate the Model on the Test Set ---
print("\nEvaluating the model on the test set...")
loss, mae = model.evaluate(X_test_scaled, y_test, verbose=0)
print(f"Test Mean Absolute Error: {mae:.4f}")
print(f"This means the model's predictions for log_price are, on average, off by {mae:.4f}.")


# --- Step 8: Make Predictions ---
print("\nMaking predictions on a few samples...")
predictions = model.predict(X_test_scaled[:5]).flatten() # Get predictions for the first 5 test samples

# Create a DataFrame to compare actual vs. predicted values
comparison_df = pd.DataFrame({
    'Actual log_price': y_test.iloc[:5].values,
    'Predicted log_price': predictions
})
print(comparison_df)

In [ ]:
import numpy as np

def smape(y_true, y_pred):
    """
    Calculates the Symmetric Mean Absolute Percentage Error (SMAPE).
    
    This function is robust against division-by-zero errors that occur
    when both the true and predicted values are zero.
    
    Args:
        y_true (array-like): Array of actual, ground-truth values.
        y_pred (array-like): Array of predicted values.
        
    Returns:
        float: The SMAPE score (as a decimal, e.g., 0.1 for 10%).
    """
    # Convert inputs to numpy arrays to ensure element-wise operations
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Calculate the numerator and denominator for the SMAPE formula
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    
    # Calculate the element-wise percentage error
    # Use np.where to handle the case where the denominator is zero
    # (which happens if y_true and y_pred are both 0 for a sample)
    # In that case, the error for that sample is defined as 0.
    ratio = np.where(denominator == 0, 0, numerator / denominator)
    
    # Return the average of the errors across all samples
    return np.mean(ratio)



In [ ]:
predictions = model.predict(X_test_scaled).flatten() # Get predictions for the first 5 test samples

# Create a DataFrame to compare actual vs. predicted values
comparison_df = pd.DataFrame({
    'Actual log_price': y_test.iloc[:].values,
    'Predicted log_price': predictions
})

In [ ]:
comparison_df

In [ ]:
comparison_df['Actual Price'] = np.expm1(comparison_df['Actual log_price'])
comparison_df['Predicted Price'] = np.expm1(comparison_df['Predicted log_price'])


In [ ]:
comparison_df

In [ ]:

# --- Example Usage ---

# Sample actual and predicted values
actual_prices = comparison_df['Actual Price'].tolist()
predicted_prices = comparison_df['Predicted Price'].tolist()

# Calculate the SMAPE score
smape_score = smape(actual_prices, predicted_prices)

print(f"The SMAPE score is: {smape_score:.4f}")
print(f"As a percentage, the error is: {smape_score * 100:.2f}%")

